In [1]:
from torch_UNet_A_D_AS import UNet_A_D_ASPP
from torch_UNet import UNet
from torch_UNet_D import UNet_D
from torch_UNet_A import UNet_A
from torch_UNet_A_D import UNet_A_D
import torch
from skimage.transform import resize
from keras.utils import load_img, img_to_array
from tqdm.notebook import tqdm
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
# === Гиперпараметры ===
im_height, im_width = 256, 256  # поменяй под нужный размер
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
DEVICE = torch.device("cuda")

In [3]:
# === DICE LOSS ===
def dice_loss(pred, target, smooth=1.):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=2).sum(dim=2)
    dice = (2. * intersection + smooth) / (
        pred.sum(dim=2).sum(dim=2) + target.sum(dim=2).sum(dim=2) + smooth
    )
    return 1 - dice.mean()

In [4]:
# === DATA LOADER ===
class TiandituDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].transpose(2, 0, 1)  # HWC -> CHW
        mask = self.masks[idx].transpose(2, 0, 1)
        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [5]:
# === ЗАГРУЗКА ДАННЫХ ===
def get_data(path, train=True):
    ids = next(os.walk(path + "images"))[2]
    #ids = ids[:100]
    X = np.zeros((len(ids), im_height, im_width, 1), dtype=np.float32)
    if train:
        y = np.zeros((len(ids), im_height, im_width, 1), dtype=np.float32)

    print('Getting and resizing images ... ')
    for n, id_ in tqdm(enumerate(ids), total=len(ids)):
        img = load_img(path + 'images/' + id_, grayscale=True)
        x_img = img_to_array(img)
        x_img = resize(x_img, (im_height, im_width, 1), mode='constant', preserve_range=True)

        fname, extension = os.path.splitext(id_)
        mask_id_ = fname + '.jpg' if extension != '.jpg' else id_

        if train:
            mask = img_to_array(load_img(path + 'masks/' + mask_id_, grayscale=True))
            mask = resize(mask, (im_height, im_width, 1), mode='constant', preserve_range=True)

        X[n, ..., 0] = x_img.squeeze() / 255
        if train:
            y[n] = mask / 255
    print('Done!')
    return (X, y) if train else X

In [7]:
# === Загрузка numpy-данных ===
X_np, y_np = get_data('../training_dataset/tianditu/', train=True)

Getting and resizing images ... 


  0%|          | 0/1190 [00:00<?, ?it/s]

Done!


In [8]:
X_new, y_new = get_data('C:/Users/nebrosarth/Documents/Files/thesis/Dataset/new_method/', train=True)

Getting and resizing images ... 


  0%|          | 0/5000 [00:00<?, ?it/s]

Done!


In [9]:
X_old, y_old = get_data('C:/Users/nebrosarth/Documents/Files/thesis/Dataset/old_method/', train=True)

Getting and resizing images ... 


  0%|          | 0/5000 [00:00<?, ?it/s]

Done!


In [11]:
X_total = np.concatenate((X_np, X_new, X_old), axis=0)
y_total = np.concatenate((y_np, y_new, y_old), axis=0)

In [12]:
print(len(X_total), len(y_total))

11190 11190


In [13]:
# === Разделение на train/val ===
X_train, X_val, y_train, y_val = train_test_split(X_total, y_total, test_size=0.2, random_state=42)

In [29]:
# === Torch Dataset/DataLoader ===
train_dataset = TiandituDataset(X_train, y_train)
val_dataset = TiandituDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [32]:
# === Инициализация модели ===
model = UNet_A_D_ASPP(in_channels=1).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_loss = float('inf')

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [36]:
# === ОБУЧЕНИЕ ===
for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    for images, masks in train_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        outputs = model(images)
        loss = dice_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.detach().item()

    avg_train_loss = epoch_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            outputs = model(images)
            loss = dice_loss(outputs, masks)

            val_loss += loss.detach().item()

    avg_val_loss = val_loss / len(val_loader)
    
    scheduler.step(avg_val_loss)

    # --- Сохранение лучшей модели ---
    new_record = avg_val_loss < best_val_loss
    if new_record:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), f'{model.__class__.__name__}.pth')

    # --- Лог ---
    print(f"Epoch [{epoch+1}/{EPOCHS}]  Train Loss: {avg_train_loss:.4f}  Val Loss: {avg_val_loss:.4f} {'+' if new_record else ''}")

Epoch [1/20]  Train Loss: 0.1250  Val Loss: 0.1220 
Epoch [2/20]  Train Loss: 0.1249  Val Loss: 0.1215 +
Epoch [3/20]  Train Loss: 0.1245  Val Loss: 0.1217 
Epoch [4/20]  Train Loss: 0.1246  Val Loss: 0.1210 +
Epoch [5/20]  Train Loss: 0.1243  Val Loss: 0.1211 
Epoch [6/20]  Train Loss: 0.1241  Val Loss: 0.1208 +
Epoch [7/20]  Train Loss: 0.1242  Val Loss: 0.1205 +
Epoch [8/20]  Train Loss: 0.1240  Val Loss: 0.1207 
Epoch [9/20]  Train Loss: 0.1240  Val Loss: 0.1216 
Epoch [10/20]  Train Loss: 0.1237  Val Loss: 0.1204 +
Epoch [11/20]  Train Loss: 0.1237  Val Loss: 0.1207 
Epoch [12/20]  Train Loss: 0.1235  Val Loss: 0.1203 +
Epoch [13/20]  Train Loss: 0.1233  Val Loss: 0.1203 
Epoch [14/20]  Train Loss: 0.1232  Val Loss: 0.1202 +
Epoch [15/20]  Train Loss: 0.1234  Val Loss: 0.1205 
Epoch [16/20]  Train Loss: 0.1235  Val Loss: 0.1201 +
Epoch [17/20]  Train Loss: 0.1233  Val Loss: 0.1203 
Epoch [18/20]  Train Loss: 0.1232  Val Loss: 0.1200 +
Epoch [19/20]  Train Loss: 0.1231  Val Loss: 0

In [34]:
#torch.save(model.state_dict(), "model.pth")